# House Prices — EDA

Этот ноутбук показывает компактный, но осмысленный EDA для соревнования `House Prices - Advanced Regression Techniques`.
Здесь мы проверяем форму таргета, характер пропусков и связь числовых признаков с ценой, чтобы затем осознанно выбирать признаки и модели.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

# Загружаем данные локально из папки проекта, чтобы ноутбук можно было открыть отдельно.
data_dir = Path("../data")
train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")

display(train.head())
print(f"train shape: {train.shape}")
print(f"test shape: {test.shape}")
print(f"target mean: {train['SalePrice'].mean():.0f}")
print(f"target median: {train['SalePrice'].median():.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(train["SalePrice"], bins=40, kde=True, ax=axes[0], color="#2a6f97")
axes[0].set_title("Распределение SalePrice")
axes[0].set_xlabel("SalePrice")

sns.histplot(np.log1p(train["SalePrice"]), bins=40, kde=True, ax=axes[1], color="#ff7f0e")
axes[1].set_title("Распределение log1p(SalePrice)")
axes[1].set_xlabel("log1p(SalePrice)")

plt.tight_layout()

In [ ]:
missing = train.isna().mean().sort_values(ascending=False).head(15)
num_cols = train.select_dtypes(include=["number"]).columns
corr = train[num_cols].corr(numeric_only=True)["SalePrice"].sort_values(ascending=False)
strong_corr = corr.drop("SalePrice").head(12)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(x=missing.values, y=missing.index, ax=axes[0], color="#6c757d")
axes[0].set_title("Топ-15 столбцов по доле пропусков")
axes[0].set_xlabel("Доля пропусков")
axes[0].set_ylabel("Столбец")

sns.barplot(x=strong_corr.values, y=strong_corr.index, ax=axes[1], color="#2f855a")
axes[1].set_title("Связь числовых признаков с SalePrice")
axes[1].set_xlabel("Корреляция с SalePrice")
axes[1].set_ylabel("Признак")

plt.tight_layout()

## Короткие выводы

- Таргет `SalePrice` заметно скошен вправо, поэтому в пайплайне используется `log1p(SalePrice)` для стабилизации обучения.
- Часть признаков имеет сильную категориальную природу: материал, район, состояние и качество дома важнее интерпретировать через кодирование, а не как голые числа.
- Пропуски концентрируются неравномерно: для части колонок уместнее индикаторы и аккуратная импутация, чем простое удаление столбцов.

Ниже добавлены два графика: распределение таргета в исходном и логарифмированном масштабе, а также связь числовых признаков с `SalePrice`.